In [49]:
!pip install groq pandas

In [50]:
import pandas as pd
import json
from groq import Groq

In [51]:
client = Groq(
      api_key="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"
      )

In [52]:
messy_invoices = [
      "Invoice #101 John bought Laptop for $800 on 12 Jan 2025",
       "Inv-102 | Sarah | Mobile | Rs.25000 | 15-Feb-2025",
       "Bill No:103 Customer: Mike Product: Headphones Amount:$120 Date:2025/02/20",
       "Invoice104 Emma purchased Tablet costing 300 dollars on March 1 2025",
       "INV105 David, Smart Watch, ₹5000, 05-03-2025"
                      ]


In [53]:
def extract_invoice(invoice_text):
    prompt = f"""
        Extract the following fields from the invoice:

            invoice_id
            customer_name
            product
            amount
            date
        Return ONLY valid JSON.
        Invoice:
            {invoice_text}
    """
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )
    result = response.choices[0].message.content
    # Remove markdown code block wrappers if present
    if result.startswith('```json') and result.endswith('```'):
        result = result[len('```json'):-len('```')].strip()

    return json.loads(result)

In [54]:
records = []

for invoice in messy_invoices:
    data = extract_invoice(invoice)
    records.append(data)

In [55]:
for record in records:
      print(record)

{'invoice_id': '101', 'customer_name': 'John', 'product': 'Laptop', 'amount': 800, 'date': '12 Jan 2025'}
{'invoice_id': 'Inv-102', 'customer_name': 'Sarah', 'product': 'Mobile', 'amount': 'Rs.25000', 'date': '15-Feb-2025'}
{'invoice_id': '103', 'customer_name': 'Mike', 'product': 'Headphones', 'amount': 120, 'date': '2025-02-20'}
{'invoice_id': 'Invoice104', 'customer_name': 'Emma', 'product': 'Tablet', 'amount': 300, 'date': 'March 1, 2025'}
{'invoice_id': 'INV105', 'customer_name': 'David', 'product': 'Smart Watch', 'amount': 5000, 'date': '05-03-2025'}


In [56]:
df = pd.DataFrame(records)

In [57]:
print(df)

   invoice_id customer_name      product    amount           date
0         101          John       Laptop       800    12 Jan 2025
1     Inv-102         Sarah       Mobile  Rs.25000    15-Feb-2025
2         103          Mike   Headphones       120     2025-02-20
3  Invoice104          Emma       Tablet       300  March 1, 2025
4      INV105         David  Smart Watch      5000     05-03-2025


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   invoice_id     5 non-null      object
 1   customer_name  5 non-null      object
 2   product        5 non-null      object
 3   amount         5 non-null      object
 4   date           5 non-null      object
dtypes: object(5)
memory usage: 332.0+ bytes


In [59]:
print(df.isnull().sum())

invoice_id       0
customer_name    0
product          0
amount           0
date             0
dtype: int64


In [60]:
df['amount'] = df['amount'].astype(str).str.replace(r'\D', '', regex=True)
df['amount'] = pd.to_numeric(df['amount'])

In [61]:
def parse_dates_robustly(date_str):
    formats_to_try = [
        '%d %b %Y',      # "12 Jan 2025"
        '%d-%b-%Y',      # "15-Feb-2025"
        '%Y-%m-%d',      # "2025-02-20"
        '%B %d, %Y',     # "March 1, 2025"
        '%d-%m-%Y',      # "05-03-2025" (day-month-year)
        '%m-%d-%Y',      # "05-03-2025" (month-day-year)
    ]
    for fmt in formats_to_try:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except ValueError:
            continue
    return pd.NaT # Return Not a Time for unparseable dates

df['date'] = df['date'].apply(parse_dates_robustly)

In [62]:
print(df.describe())

             amount                 date
count      5.000000                    5
mean    6244.000000  2025-02-15 14:24:00
min      120.000000  2025-01-12 00:00:00
25%      300.000000  2025-02-15 00:00:00
50%      800.000000  2025-02-20 00:00:00
75%     5000.000000  2025-03-01 00:00:00
max    25000.000000  2025-03-05 00:00:00
std    10674.815221                  NaN


In [63]:
print("Total Revenue:", df['amount'].sum())

Total Revenue: 31220


In [64]:
print("Average Invoice Value:", df['amount'].mean())

Average Invoice Value: 6244.0


In [65]:
print(df.loc[df['amount'].idxmax()])

invoice_id                   Inv-102
customer_name                  Sarah
product                       Mobile
amount                         25000
date             2025-02-15 00:00:00
Name: 1, dtype: object


In [66]:
print(df.groupby('product')['amount'].sum())

product
Headphones       120
Laptop           800
Mobile         25000
Smart Watch     5000
Tablet           300
Name: amount, dtype: int64


In [67]:
df.to_csv("clean_invoice_data.csv", index=False)

In [68]:
print("="*50)
print("SMART DATA CLEANER - FINAL REPORT")
print("="*50)

print(f"Total Invoices Processed : {len(df)}")
print(f"Total Revenue           : {df['amount'].sum()}")
print(f"Average Invoice Value   : {df['amount'].mean():.2f}")
print(f"Highest Invoice Amount  : {df['amount'].max()}")

print("\nProduct-wise Revenue")
print(df.groupby('product')['amount'].sum())

print("\nData Cleaning Status")
print("Missing Values:")
print(df.isnull().sum())

print("="*50)
print("ETL PIPELINE COMPLETED SUCCESSFULLY")
print("="*50)

SMART DATA CLEANER - FINAL REPORT
Total Invoices Processed : 5
Total Revenue           : 31220
Average Invoice Value   : 6244.00
Highest Invoice Amount  : 25000

Product-wise Revenue
product
Headphones       120
Laptop           800
Mobile         25000
Smart Watch     5000
Tablet           300
Name: amount, dtype: int64

Data Cleaning Status
Missing Values:
invoice_id       0
customer_name    0
product          0
amount           0
date             0
dtype: int64
ETL PIPELINE COMPLETED SUCCESSFULLY
